# Makemore MLP — E01 기준 모델 재구성

다른 컴퓨터에서 학습·평가 구현, 학습률을 0.01로 낮춘 뒤 약 20,000회 학습까지 진행했다는 학습자의 설명을 바탕으로, 명시적 예외 요청에 따라 AI가 실행 준비 코드를 재구성했다.

당시 코드·출력·가중치의 복구본은 아니다. 배치 32와 E02 초기화는 현재 보관본을 기반으로 한 재구성 설정이다. 학습률을 낮추기 전의 가중치와 학습 횟수는 확인되지 않았다. 따라서 여기서는 새 초기화에서 학습률 0.01로 20,000회 실행하도록 준비했다. 당시의 이어 학습과는 출발 가중치가 다르다. 아래 실행 결과는 이 컴퓨터의 새 실험이다.

프로젝트 `.venv` 커널에서 위에서부터 실행한다. 초기화 셀을 다시 실행하면 모델과 학습 기록이 초기화된다. 학습 셀을 다시 실행하면 현재 모델에서 추가 학습한다. test는 튜닝에 사용하지 않는다.

참고: [공식 구현](https://github.com/karpathy/nn-zero-to-hero/blob/master/lectures/makemore/makemore_part2_mlp.ipynb), `practice/deep-learning/makemore-mlp-e02-initialization-training.ipynb`.

In [1]:
import random
from urllib.request import urlopen

import torch
import torch.nn.functional as F

with urlopen("https://raw.githubusercontent.com/karpathy/makemore/master/names.txt", timeout=30) as response:
    words = response.read().decode("utf-8").splitlines()

chars = sorted(set("".join(words)))
stoi = {ch: i + 1 for i, ch in enumerate(chars)}
stoi["."] = 0
itos = {i: ch for ch, i in stoi.items()}
vocab_size = len(stoi)
block_size = 3
embedding_dim = 10
hidden_size = 200
batch_size = 32
device = torch.device("cpu")
print("words:", len(words), "vocabulary:", vocab_size, "device:", device)

words: 32033 vocabulary: 27 device: cpu


In [2]:
def build_dataset(word_list):
    X, Y = [], []
    for word in word_list:
        context = [0] * block_size
        for ch in word + ".":
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    return (
        torch.tensor(X, dtype=torch.long, device=device),
        torch.tensor(Y, dtype=torch.long, device=device),
    )

shuffled_words = words.copy()
random.Random(42).shuffle(shuffled_words)
n = len(shuffled_words)
train_end, dev_end = n * 8 // 10, n * 9 // 10
Xtr, Ytr = build_dataset(shuffled_words[:train_end])
Xdev, Ydev = build_dataset(shuffled_words[train_end:dev_end])
Xte, Yte = build_dataset(shuffled_words[dev_end:])
for name, X, Y in [("train", Xtr, Ytr), ("dev", Xdev, Ydev), ("test", Xte, Yte)]:
    print(name, X.shape, Y.shape)

train torch.Size([182625, 3]) torch.Size([182625])
dev torch.Size([22655, 3]) torch.Size([22655])
test torch.Size([22866, 3]) torch.Size([22866])


In [3]:
# 다시 실행하면 모델·난수 시퀀스·기록을 초기화합니다.
g_model = torch.Generator(device=device).manual_seed(2147483647)
g_batch = torch.Generator(device=device).manual_seed(42)
C = torch.randn((vocab_size, embedding_dim), generator=g_model, device=device, dtype=torch.float32)
W1 = torch.randn((block_size * embedding_dim, hidden_size), generator=g_model, device=device, dtype=torch.float32)
b1 = torch.randn(hidden_size, generator=g_model, device=device, dtype=torch.float32)
W2 = torch.randn((hidden_size, vocab_size), generator=g_model, device=device, dtype=torch.float32) * 0.02
b2 = torch.zeros(vocab_size, device=device, dtype=torch.float32)
parameters = [C, W1, b1, W2, b2]
for p in parameters:
    p.requires_grad_(True)

steps_completed = 0
loss_history = []
evaluations = []
print("parameters:", sum(p.numel() for p in parameters))

parameters: 11897


In [4]:
def forward(X):
    emb = C[X]
    flat = emb.flatten(start_dim=1)
    h = torch.tanh(flat @ W1 + b1)
    return h @ W2 + b2

@torch.no_grad()
def evaluate(X, Y, chunk_size=4096):
    total_loss = 0.0
    for start in range(0, len(Y), chunk_size):
        logits = forward(X[start:start + chunk_size])
        total_loss += F.cross_entropy(
            logits, Y[start:start + chunk_size], reduction="sum"
        ).item()
    return total_loss / len(Y)

def record_evaluation(label):
    result = {
        "label": label,
        "step": steps_completed,
        "train_loss": evaluate(Xtr, Ytr),
        "dev_loss": evaluate(Xdev, Ydev),
    }
    evaluations.append(result)
    print(result)

def train_steps(num_steps, learning_rate, log_every=1000):
    global steps_completed
    for _ in range(num_steps):
        indices = torch.randint(len(Xtr), (batch_size,), generator=g_batch, device=device)
        loss = F.cross_entropy(forward(Xtr[indices]), Ytr[indices])
        for p in parameters:
            p.grad = None
        loss.backward()
        with torch.no_grad():
            for p in parameters:
                p -= learning_rate * p.grad
        steps_completed += 1
        loss_history.append((steps_completed, learning_rate, loss.item()))
        if steps_completed % log_every == 0:
            print(f"step={steps_completed}, lr={learning_rate}, batch loss (before update)={loss.item():.4f}")

record_evaluation("initial")

{'label': 'initial', 'step': 0, 'train_loss': 3.3347315756545175, 'dev_loss': 3.334264605978261}


In [5]:
# 당시 lr 변경 전 모델은 없으므로, 여기서는 새 초기화부터 실행합니다.
# 재실행하면 현재 모델에서 20,000회 추가 학습합니다.
lr = 0.01
train_steps(20_000, learning_rate=lr)
record_evaluation("after 20,000 steps at lr=0.01")

step=1000, lr=0.01, batch loss (before update)=2.8690
step=2000, lr=0.01, batch loss (before update)=2.2701
step=3000, lr=0.01, batch loss (before update)=2.8083
step=4000, lr=0.01, batch loss (before update)=2.4742
step=5000, lr=0.01, batch loss (before update)=2.0964
step=6000, lr=0.01, batch loss (before update)=2.3979
step=7000, lr=0.01, batch loss (before update)=2.1656
step=8000, lr=0.01, batch loss (before update)=2.1439
step=9000, lr=0.01, batch loss (before update)=2.2047
step=10000, lr=0.01, batch loss (before update)=2.3678
step=11000, lr=0.01, batch loss (before update)=1.8799
step=12000, lr=0.01, batch loss (before update)=2.1181
step=13000, lr=0.01, batch loss (before update)=2.4439
step=14000, lr=0.01, batch loss (before update)=2.1169
step=15000, lr=0.01, batch loss (before update)=2.1333
step=16000, lr=0.01, batch loss (before update)=2.1830
step=17000, lr=0.01, batch loss (before update)=2.2137
step=18000, lr=0.01, batch loss (before update)=2.1293
step=19000, lr=0.01

In [6]:
print("learning rate:", lr, "completed steps:", steps_completed)

learning rate: 0.01 completed steps: 20000


## 기준 결과 확인 후 다음 설정 비교

학습자는 다른 컴퓨터에서 학습률을 0.01로 낮춘 뒤 약 20,000회 학습했다고 설명했다. 위 셀은 그 학습률과 횟수를 반영하지만 당시의 학습된 가중치에서 이어지는 실행은 아니다.

아래 추가 학습은 기본값 0으로 두었다. 기준 train/dev 결과를 확인한 뒤 다음 비교 설정을 결정한다. 이전 에이전트가 제안한 변경 항목은 현재 기록에 없어 특정 항목으로 단정하지 않는다. 추가 학습이 필요할 때만 `additional_steps`를 지정한다.

In [ ]:
additional_steps = 20000
if additional_steps > 0:
    train_steps(additional_steps, learning_rate=lr)
    record_evaluation("after lr=0.01 continuation")
else:
    print("추가 학습 대기: additional_steps를 지정하세요.")

추가 학습 대기: additional_steps를 지정하세요.


In [8]:
# 이번 커널에서 실제 평가한 결과만 표시합니다.
for result in evaluations:
    print(result)

{'label': 'initial', 'step': 0, 'train_loss': 3.3347315756545175, 'dev_loss': 3.334264605978261}
{'label': 'after 20,000 steps at lr=0.01', 'step': 20000, 'train_loss': 2.283116345290041, 'dev_loss': 2.291656953487089}
